# 🌿 Plant Disease Detection — Google Colab Training Notebook

**Self-contained** — no local files needed. Run cells top to bottom.

### Steps
1. Connect to a GPU runtime: `Runtime → Change runtime type → T4 GPU`
2. Run all cells in order
3. Model + class names will be saved to your Google Drive
4. Download them and place in your local `models/` folder

**Expected training time on T4 GPU**: ~35–50 minutes (20 epochs, 87K images)

## Cell 1 — Check GPU

In [ ]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime -> Change runtime type -> T4 GPU')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/PlantDisease_Models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Model will be saved to: {SAVE_DIR}')

## Cell 3 — Install Kaggle & Download Dataset

In [ ]:
# Install kaggle
!pip install kaggle -q

# Save the Kaggle token
import os, json
os.makedirs('/root/.kaggle', exist_ok=True)

# ── PASTE YOUR KAGGLE TOKEN BELOW ──────────────────────────────────────────
# Option A: If you have kaggle.json content, paste it here:
# kaggle_creds = {"username": "your_username", "key": "your_key"}
# with open('/root/.kaggle/kaggle.json', 'w') as f:
#     json.dump(kaggle_creds, f)
# os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Option B: If you have the KGAT_ token (new format), use this:
ACCESS_TOKEN = 'KGAT_2d3c8e53189ed67a2df14d817c5839ef'  # <-- your token
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(ACCESS_TOKEN)
# ────────────────────────────────────────────────────────────────────────────

print('Kaggle credentials saved.')

In [ ]:
# Download & unzip dataset (~3.5 GB, ~4 minutes on Colab)
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content/data --unzip
print('Download complete.')

In [ ]:
# Locate the actual train/ and valid/ folders (nested structure)
import shutil
from pathlib import Path

DATA_ROOT = Path('/content/data/plantvillage')
DATA_ROOT.mkdir(exist_ok=True)

# Find where train/ and valid/ actually landed
source = None
for root, dirs, _ in os.walk('/content/data'):
    r = Path(root)
    if (r / 'train').exists() and (r / 'valid').exists():
        source = r
        break

if source and source != DATA_ROOT:
    for split in ('train', 'valid'):
        dst = DATA_ROOT / split
        if not dst.exists():
            shutil.move(str(source / split), str(dst))
            print(f'Moved {split}/')
elif source == DATA_ROOT:
    print('Already in correct location.')

# Count classes
for split in ('train', 'valid'):
    n = len(list((DATA_ROOT / split).iterdir()))
    print(f'{split}: {n} classes')

In [ ]:
# Create test/ split (50% of valid/)
import random
random.seed(42)

test_dir = DATA_ROOT / 'test'
class_names = sorted([d.name for d in (DATA_ROOT / 'train').iterdir() if d.is_dir()])
print(f'Classes: {len(class_names)}')

if not test_dir.exists() or not any(test_dir.iterdir()):
    for cls in class_names:
        valid_cls = DATA_ROOT / 'valid' / cls
        test_cls  = test_dir  / cls
        test_cls.mkdir(parents=True, exist_ok=True)
        images = [f for f in valid_cls.iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
        for img in random.sample(images, len(images) // 2):
            shutil.move(str(img), str(test_cls / img.name))
    print('Test split created.')
else:
    print('Test split already exists.')

for split in ('train', 'valid', 'test'):
    total = sum(len(list((DATA_ROOT / split / c).iterdir())) for c in class_names)
    print(f'{split}: {total:,} images')

## Cell 4 — Imports & Config

In [ ]:
import json, warnings, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from torchvision.models import ResNet50_Weights
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_ROOT   = Path('/content/data/plantvillage')
MODELS_DIR  = Path('/content/models')
FIGURES_DIR = Path('/content/figures')
MODELS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

CHECKPOINT = MODELS_DIR / 'plant_disease_resnet50.pth'

# ── Hyperparameters ──────────────────────────────────────────────────────────
NUM_CLASSES = 38
BATCH_SIZE  = 64    # T4 can handle 64 easily
MAX_EPOCHS  = 20
EARLY_STOP  = 5

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Style
plt.style.use('dark_background')
TEAL, ORANGE = '#00d4aa', '#f57935'

print('Config ready.')

## Cell 5 — Transforms & DataLoaders

In [ ]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

infer_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(DATA_ROOT / 'train', transform=train_transforms)
valid_ds = datasets.ImageFolder(DATA_ROOT / 'valid', transform=infer_transforms)
test_ds  = datasets.ImageFolder(DATA_ROOT / 'test',  transform=infer_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class_names = train_ds.classes
assert len(class_names) == NUM_CLASSES, f'Expected {NUM_CLASSES} classes, got {len(class_names)}'

print(f'Train: {len(train_ds):,} | Valid: {len(valid_ds):,} | Test: {len(test_ds):,}')
print(f'Classes: {len(class_names)}')

# Save class names
with open(MODELS_DIR / 'class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)
print('class_names.json saved.')

## Cell 6 — EDA Plots

In [ ]:
# Class distribution
train_counts = [len(list((DATA_ROOT / 'train' / c).iterdir())) for c in class_names]
human_names  = [c.replace('___', '\n').replace('_', ' ') for c in class_names]

fig, ax = plt.subplots(figsize=(16, 7))
colors = [ORANGE if 'healthy' in c else TEAL for c in class_names]
ax.barh(human_names, train_counts, color=colors, alpha=0.85)
ax.set_title('Class Distribution — Teal = Diseased, Orange = Healthy', fontsize=13)
ax.set_xlabel('Training Images')
ax.tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Min: {min(train_counts)} | Max: {max(train_counts)} | Mean: {int(sum(train_counts)/len(train_counts))}')

In [ ]:
# 4x4 sample image grid
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for ax, cls in zip(axes.flat, random.sample(class_names, 16)):
    img = Image.open(random.choice(list((DATA_ROOT / 'train' / cls).iterdir()))).convert('RGB').resize((224,224))
    ax.imshow(img)
    ax.set_title(cls.replace('___','\n').replace('_',' '), fontsize=6,
                 color=ORANGE if 'healthy' in cls else TEAL)
    ax.axis('off')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'sample_grid.png', dpi=120, bbox_inches='tight')
plt.show()

## Cell 7 — Build Model

In [ ]:
model = models.resnet50(weights=ResNet50_Weights.DEFAULT)

# Freeze all layers
for p in model.parameters():
    p.requires_grad = False

# Unfreeze layer4
for p in model.layer4.parameters():
    p.requires_grad = True

# Replace FC head
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,} ({100*trainable/total:.1f}%)')

## Cell 8 — Train

In [ ]:
optimizer = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.fc.parameters(),     'lr': 1e-3},
])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.1, verbose=True)
criterion = nn.CrossEntropyLoss()

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
no_improve    = 0

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in tqdm(loader, desc='Train' if train else 'Valid', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train: optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, labels)
            if train: loss.backward(); optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            correct    += (out.argmax(1) == labels).sum().item()
            total      += imgs.size(0)
    return total_loss / total, correct / total


print(f'Training on {DEVICE} | Batch {BATCH_SIZE} | Max {MAX_EPOCHS} epochs\n')

for epoch in range(1, MAX_EPOCHS + 1):
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(valid_loader, train=False)
    scheduler.step(vl)

    history['train_loss'].append(tl); history['val_loss'].append(vl)
    history['train_acc'].append(ta);  history['val_acc'].append(va)

    print(f'Epoch {epoch:02d}/{MAX_EPOCHS} | '
          f'Train Loss {tl:.4f}  Acc {ta*100:.2f}% | '
          f'Val Loss {vl:.4f}  Acc {va*100:.2f}%')

    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(model.state_dict(), CHECKPOINT)
        print(f'  --> Best model saved  (val_loss={best_val_loss:.4f})')
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= EARLY_STOP:
            print(f'Early stopping at epoch {epoch}'); break

print(f'\nDone. Best val_loss: {best_val_loss:.4f}')

## Cell 9 — Training Curves

In [ ]:
ep = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(ep, history['train_loss'], TEAL,   lw=2, label='Train')
ax1.plot(ep, history['val_loss'],   ORANGE, lw=2, label='Val', linestyle='--')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(alpha=0.2); ax1.set_xlabel('Epoch')

ax2.plot(ep, [a*100 for a in history['train_acc']], TEAL,   lw=2, label='Train')
ax2.plot(ep, [a*100 for a in history['val_acc']],   ORANGE, lw=2, label='Val', linestyle='--')
ax2.set_title('Accuracy (%)'); ax2.legend(); ax2.grid(alpha=0.2); ax2.set_xlabel('Epoch')

plt.suptitle('Training Curves', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## Cell 10 — Evaluate on Test Set

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='Testing'):
        preds = model(imgs.to(DEVICE)).argmax(1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

test_acc = sum(p==l for p,l in zip(all_preds, all_labels)) / len(all_labels)
print(f'Test Accuracy: {test_acc*100:.2f}%\n')
print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
short = [c.split('___')[-1].replace('_',' ')[:16] for c in class_names]

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(cm, annot=False, xticklabels=short, yticklabels=short,
            cmap='YlOrRd', ax=ax, linewidths=0.2)
ax.set_title('Confusion Matrix — Test Set', fontsize=15)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.tick_params(axis='x', rotation=90, labelsize=7)
ax.tick_params(axis='y', rotation=0,  labelsize=7)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

## Cell 11 — Grad-CAM Visualisation

In [ ]:
import cv2, numpy as np, torch.nn.functional as F

class GradCAM:
    def __init__(self, model):
        self.model = model.eval()
        self._grad = self._act = None
        model.layer4[-1].register_forward_hook(lambda m,i,o: setattr(self,'_act',o.detach()))
        model.layer4[-1].register_full_backward_hook(lambda m,gi,go: setattr(self,'_grad',go[0].detach()))

    def generate(self, tensor, class_idx=None):
        self.model.zero_grad()
        out = self.model(tensor)
        if class_idx is None: class_idx = out.argmax(1).item()
        out[0, class_idx].backward()
        weights = self._grad.mean(dim=(2,3), keepdim=True)
        cam = F.relu((weights * self._act).sum(1, keepdim=True))
        cam = F.interpolate(cam, (224,224), mode='bilinear', align_corners=False).squeeze().cpu().numpy()
        cam -= cam.min(); cam /= (cam.max() + 1e-8)
        return cam.astype(np.float32)

    def overlay(self, pil_img, heatmap, alpha=0.45):
        img = np.array(pil_img.convert('RGB').resize((224,224)))
        col = cv2.applyColorMap((heatmap*255).astype(np.uint8), cv2.COLORMAP_JET)
        col = cv2.cvtColor(col, cv2.COLOR_BGR2RGB)
        return Image.fromarray((alpha*col + (1-alpha)*img).astype(np.uint8))


cam = GradCAM(model)
sample_classes = random.sample(class_names, 6)

fig, axes = plt.subplots(6, 3, figsize=(10, 22))
fig.suptitle('Grad-CAM — Model Attention on Leaf Regions', fontsize=13)

for row, cls in enumerate(sample_classes):
    img_path = random.choice(list((DATA_ROOT / 'test' / cls).iterdir()))
    pil_img  = Image.open(img_path).convert('RGB')

    tensor = infer_transforms(pil_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): probs = torch.softmax(model(tensor), 1)
    pred_idx  = probs.argmax(1).item()
    pred_name = class_names[pred_idx]
    conf      = probs[0, pred_idx].item()

    heatmap = cam.generate(tensor, class_idx=pred_idx)
    overlay = cam.overlay(pil_img, heatmap)

    axes[row,0].imshow(pil_img.resize((224,224)))
    axes[row,0].set_title('Original', fontsize=8); axes[row,0].axis('off')
    axes[row,1].imshow(heatmap, cmap='jet')
    axes[row,1].set_title('Heatmap', fontsize=8); axes[row,1].axis('off')
    axes[row,2].imshow(overlay)
    ok = pred_name == cls
    axes[row,2].set_title(f"{pred_name.split('___')[-1].replace('_',' ')} ({conf*100:.1f}%) {'OK' if ok else 'WRONG'}",
                          fontsize=7, color=TEAL if ok else ORANGE)
    axes[row,2].axis('off')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'gradcam_examples.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: gradcam_examples.png')

## Cell 12 — Save Everything to Google Drive

In [ ]:
import shutil

# Copy model + class names to Drive
shutil.copy(CHECKPOINT,               f'{SAVE_DIR}/plant_disease_resnet50.pth')
shutil.copy(MODELS_DIR/'class_names.json', f'{SAVE_DIR}/class_names.json')

# Copy figures to Drive
DRIVE_FIGS = f'{SAVE_DIR}/figures'
shutil.copytree(FIGURES_DIR, DRIVE_FIGS, dirs_exist_ok=True)

print(f'All files saved to Google Drive:')
print(f'  {SAVE_DIR}/')
for f in Path(SAVE_DIR).rglob('*'):
    if f.is_file():
        size = f.stat().st_size / 1e6
        print(f'    {f.name} ({size:.1f} MB)')

## Done! ✅

### What to download to your local machine

Go to **Google Drive → PlantDisease_Models/** and download:
- `plant_disease_resnet50.pth` → place in `models/`
- `class_names.json` → place in `models/`
- `figures/` folder → place in `figures/`

Then run the Streamlit app locally:
```bash
.venv\Scripts\streamlit run app.py
```